# SPEAR-Net v4 — Global artisanal-mining detection (Colab, T4)

**SPEAR-Net**: a lightweight, **spectral-prior-guided**, recall-optimized network for
detecting **artisanal & small-scale (illegal-prone) mining** in global Sentinel-2 imagery.

This notebook runs the whole v4 pipeline on a free **T4**:
clone → install → **fetch the 35.6 MB verified annotations** → fetch a **subset of 6-band
Sentinel-2** from Microsoft Planetary Computer (free, no login) → chip + cache to Drive →
build SPEAR-Net (PISP gate) → train → evaluate (per-class + area-stratified) → PISP
explainability → **leave-one-region-out** generalization.

Dataset: `SimonJasansky/mine-segmentation` (Zenodo 14195737) — 1,210 sites, verified
masks (P99.2/R95.7), with an explicit **artisanal vs industrial** label.

> **Runtime → Change runtime type → T4 GPU**, then run cells top to bottom.
> Re-run cell 2 anytime to pull the latest fixes.


## 1. Check GPU

In [ ]:
!nvidia-smi || echo "No GPU — set Runtime > Change runtime type > T4 GPU"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Clone the repository  *(re-run to pull fixes)*

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/prakhar443/illegal_mining.git"
BRANCH   = "spearnet-colab"
REPO_DIR = "illegal_mining"
if not os.path.exists(REPO_DIR):
    if subprocess.run(["git","clone","--branch",BRANCH,REPO_URL,REPO_DIR]).returncode != 0:
        subprocess.run(["git","clone",REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"fetch","origin",BRANCH])
    subprocess.run(["git","-C",REPO_DIR,"checkout",BRANCH])
    subprocess.run(["git","-C",REPO_DIR,"pull","origin",BRANCH])
%cd {REPO_DIR}
!git log --oneline -1


## 3. Install dependencies (~3 min)

Deep-learning + the geospatial stack that fetches Sentinel-2 from Planetary Computer.

In [ ]:
!pip install -q "timm>=0.9.12" "segmentation-models-pytorch>=0.3.3" ptflops scikit-learn pyyaml
!pip install -q pystac pystac-client planetary-computer stackstac rioxarray \
                geopandas rasterio shapely pyproj pyogrio tqdm
!pip install -q -e .
print("Done. If a binary import (rasterio/geopandas) fails: Runtime > Restart session, "
      "then re-run from cell 3.")


## 4. Mount Drive + set paths (fetch local, store one zip on Drive)

**Why this design:** writing thousands of chips straight to the Drive FUSE mount in a long
loop causes `ConnectionAbortedError` crashes. So we fetch to **fast local disk**
(`/content/chips`) and persist the whole dataset as **one zip on Drive**
(`spearnet_chips.zip`). Next session, we restore that single zip in seconds — no re-fetch,
and nothing huge on GitHub.


In [ ]:
import os, time
CHIPS_DIR = "/content/chips"                         # fast local disk (stable for writes)
os.makedirs(CHIPS_DIR, exist_ok=True)
DRIVE_ZIP = "/content/drive/MyDrive/spearnet_chips.zip"   # single-file persistent backup

def _mount(attempts=3):
    from google.colab import drive
    for i in range(attempts):
        try:
            drive.mount('/content/drive', force_remount=(i > 0)); return True
        except Exception as e:
            print(f"  Drive mount attempt {i+1} failed: {e}"); time.sleep(3*(i+1))
    return False

DRIVE_OK = _mount()
print("local chips ->", CHIPS_DIR, "| Drive zip ->", DRIVE_ZIP if DRIVE_OK else "(Drive unavailable)")


### 4b. Restore chips from Drive if already fetched

If you fetched in a previous session and packaged to Drive, this restores the whole dataset
locally in seconds — **skip cells 5–6 and jump to cell 7**. First time through, this is a
no-op and you continue to the fetch.


In [ ]:
import sys; sys.path.insert(0, "src")
from spearnet.data.fetch import restore_chips
if DRIVE_OK and os.path.exists(DRIVE_ZIP) and not os.path.exists(os.path.join(CHIPS_DIR, "manifest.csv")):
    restore_chips(DRIVE_ZIP, CHIPS_DIR)
    import glob
    print("restored chips:", len(glob.glob(f"{CHIPS_DIR}/*/*_img.tif")))
else:
    print("No restore needed (no Drive zip yet, or chips already present). Proceed to fetch.")


## 5. Inspect annotations (metadata only)

Resolves the real file via the **Zenodo API** (robust to version/filename changes),
downloads the 35.6 MB GeoPackage, and prints the split / mine-type / **artisanal-vs-
industrial** counts — where you confirm the ASM signal is real and trainable.


In [ ]:
import sys; sys.path.insert(0, "src")
from spearnet.data.fetch import download_annotations, print_summary

# Default resolves the file via the Zenodo API. If Zenodo is flaky, pass a direct link:
#   gpkg = download_annotations("mine_data", annot_url="https://zenodo.org/records/14195737/files/<exact_name>?download=1")
gpkg = download_annotations("mine_data")
print_summary(gpkg)


## 6. Fetch a Sentinel-2 subset → chips  *(one-time, cached to Drive)*

Pulls 6 bands (B,G,R,NIR,SWIR1,SWIR2) per tile from Planetary Computer, rasterizes the
verified masks, chips 2048→256, stores **uint16** GeoTIFFs + a `manifest.csv` carrying the
per-chip `scale` and `region`.

**Recommended 3-pass fetch** (each pass appends to the manifest; safe to re-run):
1. pilot (artisanal, tiny) to confirm everything writes,
2. all artisanal (the rare, precious class),
3. a capped industrial set for contrast.

Start with the pilot, check it writes, then raise the caps.


In [ ]:
from spearnet.data.fetch import fetch_dataset

# ---- pilot: a few artisanal tiles to confirm the fetch works ----
fetch_dataset(
    out_dir=CHIPS_DIR, annot_dir="mine_data",
    fetch_imagery=True,
    scale_filter="artisanal",
    subset_per_split=8,        # small pilot; raise later
    chip_size=256, keep_empty_frac=0.3,
)
print("\nPilot done. Inspect a chip below, then run the full passes.")


### 6b. Full fetch — RESUMABLE (just re-run this cell if the runtime crashes)

`resume=True` skips tiles already fetched/attempted, so each re-run **continues** instead
of restarting. If Colab crashes mid-fetch, simply run this cell again until it prints
`Done` with no remaining tiles. Chips go to **local disk** (stable), not Drive.


In [ ]:
# Pass 2: ALL artisanal tiles (the rare, precious class)
fetch_dataset(out_dir=CHIPS_DIR, fetch_imagery=True, scale_filter="artisanal",
              subset_per_split=None, chip_size=256, resume=True)
# Pass 3: capped industrial tiles for contrast / hard negatives (lower the cap if disk tight)
fetch_dataset(out_dir=CHIPS_DIR, fetch_imagery=True, scale_filter="industrial",
              subset_per_split=80, chip_size=256, resume=True)


### 6c. Package the fetched chips → one zip on Drive  *(do this once fetch is complete)*

Stores the whole dataset as a single file so future sessions restore it in seconds
(cell 4b) without re-fetching. Re-run after fetching more tiles to refresh the backup.


In [ ]:
from spearnet.data.fetch import package_chips
if DRIVE_OK:
    package_chips(CHIPS_DIR, DRIVE_ZIP)
    print("Backed up to Drive. Next session, cell 4b restores this automatically.")
else:
    print("Drive not mounted — re-run cell 4 to mount, then package.")


Sanity-check one fetched chip (6 bands) and its mask:

In [ ]:
import glob, rasterio, numpy as np, matplotlib.pyplot as plt
imgs = sorted(glob.glob(f"{CHIPS_DIR}/*/*_img.tif"))
print("chips written:", len(imgs))
if imgs:
    ip = imgs[len(imgs)//2]; mp = ip.replace("_img.tif", "_mask.tif")
    with rasterio.open(ip) as s: img = s.read()
    with rasterio.open(mp) as s: msk = s.read()[0]
    print("img", img.shape, "dtype", img.dtype, "| mask uniques", np.unique(msk))
    rgb = img[[2,1,0]].astype("float32")
    lo, hi = np.percentile(rgb, 2), np.percentile(rgb, 98)
    rgb = ((rgb - lo)/(hi - lo + 1e-6)).clip(0, 1)
    fig, ax = plt.subplots(1,2,figsize=(8,4))
    ax[0].imshow(rgb.transpose(1,2,0)); ax[0].set_title("RGB"); ax[1].imshow(msk); ax[1].set_title("mask")
    for a in ax: a.axis("off"); plt.show()


## 7. Config — task + how much data

`v4_binary` (mining detector, robust primary) or `v4_scale3` (artisanal vs industrial, the headline). RUN_MODE caps train chips so a run fits Colab time.

In [ ]:
import torch
from spearnet.config import load_config
from spearnet.utils import set_seed

CONFIG   = "configs/v4_scale3.yaml"   # or configs/v4_binary.yaml
RUN_MODE = "dev"                      # "smoke" | "dev" | "full"

cfg = load_config(CONFIG)
cfg.data.chips_root = CHIPS_DIR
cfg.data.manifest   = f"{CHIPS_DIR}/manifest.csv"
cfg.data.num_workers = 2
cfg.run.device = "cuda" if torch.cuda.is_available() else "cpu"

if RUN_MODE == "smoke":
    cfg.data.subset_train, cfg.data.subset_val, cfg.optim.epochs = 200, 80, 8
elif RUN_MODE == "dev":
    cfg.data.subset_train, cfg.data.subset_val, cfg.optim.epochs = None, None, 40
elif RUN_MODE == "full":
    cfg.data.subset_train, cfg.data.subset_val, cfg.optim.epochs = None, None, 60

# Persist checkpoints to Drive so a Colab disconnect doesn't lose a multi-hour run.
import os
cfg.run.out_dir = f"/content/drive/MyDrive/spearnet_runs/{os.path.basename(CONFIG).split('.')[0]}_{cfg.data.task}"
os.makedirs(cfg.run.out_dir, exist_ok=True)
# Auto-resume if a previous run left a checkpoint here.
_ckpt = os.path.join(cfg.run.out_dir, "last.pt")
cfg.run.resume = _ckpt if os.path.exists(_ckpt) else None

set_seed(cfg.run.seed)
print(f"RUN_MODE={RUN_MODE} | task={cfg.data.task} ({cfg.num_classes} classes) | "
      f"prior={cfg.model.prior_type} | bands={cfg.data.bands} | device={cfg.run.device}")
print(f"checkpoints -> {cfg.run.out_dir}" + (f"  (resuming from {_ckpt})" if cfg.run.resume else ""))


## 8. Build dataloaders

In [ ]:
from spearnet.data import build_dataloaders
loaders = build_dataloaders(cfg, splits=("train", "val"))
print("train batches:", len(loaders["train"]), "| val batches:", len(loaders["val"]))
batch = next(iter(loaders["train"]))
print({k: tuple(v.shape) for k, v in batch.items()})
print("mask classes in batch:", torch.unique(batch["mask"]).tolist())


## 9. Visualize a sample + the PISP spectral priors

NDVI (vegetation loss), MNDWI (ponds), NDTI (turbidity), BSI (bare soil/tailings).

In [ ]:
import matplotlib.pyplot as plt
from spearnet.data import compute_priors, prior_names
from spearnet.data.priors import band_index
from spearnet.utils.viz import colorize_mask, _to_hwc_uint8

img = batch["image"][0]
idx = band_index(cfg.data.band_order)
priors = compute_priors(img.unsqueeze(0), cfg.model.prior_type, idx)[0]
names = prior_names(cfg.model.prior_type)
fig, ax = plt.subplots(1, 6, figsize=(22, 4))
ax[0].imshow(_to_hwc_uint8(img)); ax[0].set_title("S2 RGB")
ax[1].imshow(colorize_mask(batch["mask"][0].numpy())); ax[1].set_title("Mask")
for i, name in enumerate(names):
    ax[i+2].imshow(priors[i].numpy(), cmap="viridis"); ax[i+2].set_title(name)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


## 10. Build SPEAR-Net + efficiency report

In [ ]:
from spearnet.models import build_model
from spearnet.utils import count_parameters, measure_efficiency
import json
model = build_model(cfg)
print("Parameters:", count_parameters(model))
print(json.dumps(measure_efficiency(model, cfg.data.image_size, cfg.run.device,
                                    in_chans=cfg.data.bands), indent=2))


## 11. Train  *(prints estimated class weights first)*

In [ ]:
from spearnet.engine import Trainer
trainer = Trainer(model, loaders, cfg)
summary = trainer.train()
print("best", cfg.run.save_best_metric, "=", summary["best_metric"])


## 12. Evaluate (per-class IoU + area-stratified recall)

In [ ]:
from spearnet.engine import evaluate
device = torch.device(cfg.run.device)
results = evaluate(model, loaders["val"], cfg, device, compute_area_stratified=True)
print(f"mIoU={results['miou']:.4f}  mF1={results['mean_f1']:.4f}  "
      f"mRecall={results['mean_recall']:.4f}  pixAcc={results['pixel_acc']:.4f}")
for cls, m in results["per_class"].items():
    print(f"  {cls:>12}: IoU={m['iou']:.3f}  recall={m['recall']:.3f}  support={m['support']}")
print("\nArea-stratified recall:", results.get("area_stratified_recall"))


## 13. Explainability — predictions + PISP attention overlay

In [ ]:
from spearnet.utils import save_prediction_panel
model.eval()
with torch.no_grad():
    vb = next(iter(loaders["val"]))
    out = model(vb["image"].to(device))
    preds = out["logits"].argmax(1).cpu()
    attn = out.get("attn")
os.makedirs("figures", exist_ok=True)
for i in range(min(3, preds.shape[0])):
    a = attn[i].cpu() if attn is not None else None
    save_prediction_panel(vb["image"][i], vb["mask"][i], preds[i],
                          f"figures/panel_{i}.png", attn=a, class_names=cfg.class_names)
from IPython.display import Image as IPImage, display
for i in range(min(3, preds.shape[0])):
    display(IPImage(f"figures/panel_{i}.png"))


## 14. Leave-one-region-out generalization (headline protocol)

Hold out whole continents from training and test on them — the publishable cross-region
result. Set `holdout_regions`, rebuild loaders (train/val now exclude those regions, and an
`ood` split appears), retrain, and compare in-region vs out-of-region.


In [ ]:
# cfg.data.holdout_regions = ["Asia", "Oceania"]
# loaders = build_dataloaders(cfg, splits=("train", "val", "ood"))
# model = build_model(cfg); trainer = Trainer(model, loaders, cfg); trainer.train()
# in_region  = evaluate(model, loaders["val"], cfg, device)["miou"]
# out_region = evaluate(model, loaders["ood"], cfg, device)["miou"]
# print(f"in-region mIoU={in_region:.3f}  out-of-region mIoU={out_region:.3f}  "
#       f"drop={in_region-out_region:.3f}")


## 15. Next steps — baselines & ablations

```python
# RGB-vs-spectral ablation (isolates the SWIR contribution of PISP):
#   model.prior_type = csp  + bands 3  vs  pisp + bands 6
# Baselines (U-Net / DeepLabV3+ / U-Net++) via the same loop:
!python scripts/train.py --config configs/v4_binary.yaml --set model.name=unet \
    data.chips_root={CHIPS_DIR} data.manifest={CHIPS_DIR}/manifest.csv
```
Switch `CONFIG` between `v4_binary.yaml` and `v4_scale3.yaml` for the detector vs the
artisanal/industrial headline.
